In [1]:
import numpy as np
import pandas as pd
pdidx = pd.IndexSlice
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from numpy import random as rand
from scipy import *
import time as T
import sys
import json
from datetime import datetime
import os

from sklearn.metrics import roc_curve, roc_auc_score, log_loss, accuracy_score, precision_recall_curve,\
                            average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import plot_importance
import optuna as opt

### In this model, we take a page out of [this arxiv paper](https://arxiv.org/pdf/2101.02118), and create a list of models, one for each time, $t$.   

In practice, this means we still train over days, but seconds_in_bucket is no longer a feature, and instead now a tree label. Therefore the amount of training data for each time is reduced by a factor of 55 (since there are 55 time isntances per day per stock). Might this make it easier to include information about the other stocks as well?
   


### Let's try first, one stock at a time   
we will also first try without the up/down classifier

## Load Data

In [ ]:
def preprocess_PoC(df, STOCK):
    df = (df
          .set_index(['date_id', 'stock_id', 'seconds_in_bucket'])
          .sort_index(level=['date_id', 'stock_id', 'seconds_in_bucket']))
    
    df = df.drop(['row_id', 'time_id'], axis=1)

    df_stock = df.loc[pd.IndexSlice[:, STOCK, :], :]
    targets_stock = df_stock[['target']]
    df_stock = df_stock.drop(['target'], axis=1)

    return df_stock, targets_stock

In [ ]:
STOCK = 168
print(f"RETAINING DATA FROM STOCK {STOCK} INTO x_wide, y_wide")
path_to_data = "../../train.csv"
ALL_DATA = pd.read_csv(path_to_data)
x_wide, y_wide = preprocess_PoC(ALL_DATA,STOCK)
%reset_selective ALL_ASSETS
x_wide = x_wide.droplevel('stock_id')
y_wide = y_wide.droplevel('stock_id')

## Construct new features

Can think about adding additional lag features.. maybe a few more target lags?

In [ ]:
def log_and_clean(df: pd.DataFrame, cols):
    df = df.copy()
    df[cols] = np.log(df[cols])
    df[cols] = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df
    
RAW_SIZE_COLS = ['bid_size', 'ask_size', 'matched_size', 'imbalance_size', 'bid_ask_lot_spread']
RAW_PRICE_COLS = ['wap', 'relative_wap']
BASE_LOG_TARGETS = RAW_SIZE_COLS + RAW_PRICE_COLS
LAG_LOG_TARGETS = [f"{c}_lag_6" for c in BASE_LOG_TARGETS]

LOG_COLS = BASE_LOG_TARGETS + LAG_LOG_TARGETS + ['wap_frac'] # wap_frac should not get lagged, it already contains a lag

### as it turns out, we don't have access to target_lag_6 when submitting. Big sad.   
### however then, we can try and use some extra proxies.
### Let's give some extra information. Maybe also lag 3. And, we give the ratio of wap/wap_lag_6

In [ ]:
# --- 1. Feature Construction ---
bid_p, ask_p = x_wide['bid_price'], x_wide['ask_price']
bid_s, ask_s = x_wide['bid_size'], x_wide['ask_size']

rel_wap = (bid_p * ask_s - ask_p * bid_s) / (bid_s + ask_s)
rel_wap.name = 'relative_wap'

log_rel_wap = rel_wap.copy()
log_rel_wap.name = 'log_relative_wap'

log_wap = x_wide['wap'].copy()
log_wap.name = 'log_wap'

BAspread = x_wide['bid_size'] - x_wide['ask_size']
BAspread.name = 'bid_ask_lot_spread'

######################################################
### Remove target_lag_6 from features

# target_lag_6 = y_wide.groupby(level=0).shift(6)
# target_lag_6 = y_wide.iloc[:, 0].groupby(level=0).shift(6)
# target_lag_6.name = 'target_lag_6'

# DY_full = y_wide.iloc[:, 0].groupby(level=0).diff(6)
# dy_lag_6 = DY_full.groupby(level=0).shift(6)
# dy_lag_6.name = 'DY_lag_6'
######################################################


# x_almost = pd.concat([x_wide, rel_wap, target_lag_6, dy_lag_6, BAspread], axis=1)
x_almost = pd.concat([x_wide, rel_wap, log_rel_wap, log_wap, BAspread], axis=1)

# --- 3. Identify features for rolling/lagging ---
# exclude = ['clf_prob', 'seconds_in_bucket', 'target_lag_6', 'DY_lag_6']
exclude = ['clf_prob', 'seconds_in_bucket']
features_to_process = [c for c in x_almost.columns if c not in exclude]

# --- 4. Rolling medians (window=6) ---
roll_df = (
    x_almost.groupby('date_id')[features_to_process]
            .rolling(window=6)
            .median()
            .reset_index(level=0, drop=True)
)
roll_df.columns = [f"{c}_roll6" for c in roll_df.columns]

# --- 5. Lag 6 only ---
lagged = (
    x_almost.groupby('date_id')[features_to_process]
            .shift(6)
            .add_suffix('_lag_6')
)

Dwap = x_almost['wap'] - lagged['wap_lag_6']
Dwap.name = 'delta_wap'

wap_frac = x_almost['wap'] / lagged['wap_lag_6']
wap_frac.name = 'wap_frac'

log_wap_frac = wap_frac.copy()
log_wap_frac.name = 'log_wap_frac'

# --- 6. Final features ---
x_new_feats = pd.concat([x_almost, Dwap, wap_frac, log_wap_frac, roll_df, lagged], axis=1).fillna(0)

# --- Log scale large quantities ---
x_new_feats = log_and_clean(x_new_feats, LOG_COLS)


encountering 0 in log is ok because we clean it up

# ---------------------------------------------------------------------------------------------------------------

# ---------------------------------------------------------------------------------------------------------------

# ---------------------------------------------------------------------------------------------------------------

# Using Optuna

In [ ]:
def export_best_params(study, using='MAE', filename="/home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/Models/Second_Model/optuna_history.txt"):
    best_params = study.best_params
    best_value = study.best_value
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # THE MODEL ACTUALLY TAKES IN A LIST OF SIZES, SO HERE WE CREATE IT AND DELETE THE REDUNDANT PARAMS
    n_bins = study.best_params['n_time_bins']
    sizes_list = []
    best_params = study.best_params
    last_size = 0
    total_seconds_used = 0
    for i in range(n_bins - 1):
        binstr = f'size_bin_{i}'
        bin_size = study.best_params[binstr]
        sizes_list.append(bin_size)
        total_seconds_used += bin_size
        best_params.pop(binstr)
    
    sizes_list.append(540 - total_seconds_used)
                          
    best_params['sizes'] = sizes_list
    if using=='MAE':
        score_string = "best_mae"
    elif using=='MSE':
        score_string = "best_MSE"
    else:
        raise ValueError('Please input MSE or MAE')
    entry = {
        "timestamp": timestamp,
        score_string: best_value,
        "params": best_params
    }
    
    with open(filename, "a") as f:
        f.write(f"STOCK NUMBER: {STOCK} \n")
        f.write(json.dumps(entry) + "\n")
        f.flush()
        os.fsync(f.fileno())
        
    print(f"--- Export Success ---")
    print(f"STOCK NUMBER: {STOCK}")
    print(f"Best params: {best_params}\n exported to {filename} at {timestamp}\n with value: {best_value}")

In [ ]:
#######################
###  Default parameters
#######################
params = {
    'N_TIME_BINS' : 8,
    'SIZES' : [120,90,90,60,60,60,30,30],
    'N_EST_BT' : 20,
    'MAX_DEPTH_BT' : 6,
    'N_EST_RT' : 1_000,
    'MAX_DEPTH_RT' : 5,
    'MIN_CHILD_WEIGHT' : 100,
    'HUBER_SLOPE' : 5,
    'BASE_SCORE_MULTIPLIER' : 1,
    'EARLY_STOPPING' : 50
}
end_clf_train = 200
end_reg_train = 400

In [ ]:
def train(params):
    TIME_STEPS = 540
    SIZES = params['SIZES']
    ENDS = np.cumsum(SIZES).tolist()
    DY_wide = y_wide - y_wide.groupby(level='date_id').shift(6)
    
    start_tracker = 0

    # total_absolute_error = 0
    # let's try square error
    total_square_error = 0
    total_samples = 0
    
    for E in ENDS:
        T_start = start_tracker
        T_end = E
        start_tracker = E
        
        # --- CLASSIFIER DATA ---
        X_clf_train = x_new_feats.loc[pdidx[:end_clf_train, T_start:T_end-1], :].reset_index().drop(columns=['date_id'])
        Y_clf_train = DY_wide.loc[pdidx[:end_clf_train, T_start:T_end-1], :].values
        # Y_clf_train = y_wide.loc[pdidx[:end_clf_train, T_start:T_end-1], :].values
    
        X_clf_valid = x_new_feats.loc[pdidx[end_clf_train+1:, T_start:T_end-1], :].reset_index().drop(columns=['date_id'])
        Y_clf_valid = DY_wide.loc[pdidx[end_clf_train+1:, T_start:T_end-1], :].values
        # Y_clf_valid = y_wide.loc[pdidx[end_clf_train+1:, T_start:T_end-1], :].values
    
        train_mask = ~np.isnan(Y_clf_train).flatten()
        valid_mask = ~np.isnan(Y_clf_valid).flatten()
        
        ytr_bin = (Y_clf_train.flatten()[train_mask] >= 0).astype(int)
        yva_bin = (Y_clf_valid.flatten()[valid_mask] >= 0).astype(int)

        # --- CLASSIFIER SAFETY ---
        if len(ytr_bin) == 0 or len(yva_bin) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Classifier data.")
            return 99999.0 # Or raise optuna.exceptions.TrialPruned()
    
        # --- FIT CLASSIFIER ---
        BT = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            learning_rate=0.05,
            n_estimators=params['N_EST_BT'],
            max_depth=params['MAX_DEPTH_BT'],
            eval_metric=["auc", "logloss"]
        )
        BT.fit(X_clf_train.iloc[train_mask], ytr_bin, 
               eval_set=[(X_clf_valid.iloc[valid_mask], yva_bin)], 
               verbose=False)
    
        # --- REGRESSOR DATA ---
        X_reg_train = x_new_feats.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end-1], :].reset_index().drop(columns=['date_id'])
        # Y_reg_train = DY_wide.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end-1], :].values
        Y_reg_train = y_wide.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end-1], :].values

        X_reg_valid = x_new_feats.loc[pdidx[end_reg_train+1:, T_start:T_end-1], :].reset_index().drop(columns=['date_id'])
        # Y_reg_valid = DY_wide.loc[pdidx[end_reg_train+1:, T_start:T_end-1], :].values
        Y_reg_valid = y_wide.loc[pdidx[end_reg_train+1:, T_start:T_end-1], :].values
    
        # Apply Classifier probabilities as a feature
        X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train)[:, 1]
        X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid)[:, 1]
    
        reg_train_mask = ~np.isnan(Y_reg_train).flatten()
        reg_valid_mask = ~np.isnan(Y_reg_valid).flatten()
        
        # --- FIT REGRESSOR ---
        y_reg_tr_clean = Y_reg_train.flatten()[reg_train_mask]
        y_reg_vld_clean = Y_reg_valid.flatten()[reg_valid_mask]

        if len(y_reg_tr_clean) == 0 or len(y_reg_vld_clean) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Regressor data.")
            return 99999.0

        current_base_score = np.mean(y_reg_tr_clean) * params['BASE_SCORE_MULTIPLIER']
        RT = xgb.XGBRegressor(
            n_estimators=params['N_EST_RT'],
            learning_rate=0.005,
            max_depth=params['MAX_DEPTH_RT'],
            min_child_weight=params['MIN_CHILD_WEIGHT'],
            objective='reg:pseudohubererror',
            huber_slope=params['HUBER_SLOPE'],
            base_score=current_base_score,
            tree_method='hist',
            early_stopping_rounds=params['EARLY_STOPPING'],
        )
    
        RT.fit(X_reg_train.iloc[reg_train_mask], y_reg_tr_clean, 
               eval_set=[(X_reg_valid.iloc[reg_valid_mask], Y_reg_valid.flatten()[reg_valid_mask])], 
               verbose=False)
        
        # --- ACCUMULATE ERROR ---
        preds = RT.predict(X_reg_valid.iloc[reg_valid_mask])
        actuals = Y_reg_valid.flatten()[reg_valid_mask]
        
        # total_absolute_error += np.sum(np.abs(preds - actuals))
        total_square_error += np.sum(np.square(preds - actuals))
        total_samples += len(actuals)
        
    #     print(f"DEBUG OPTUNA: Bin {T_start}-{T_end} | Samples: {len(actuals)} | SumAbsErr: {np.sum(np.abs(preds - actuals))}")
    # print(f"DEBUG OPTUNA: FINAL TOTAL SAMPLES: {total_samples}")

    # RETURN SQAURED ERROR SCORE!!!!!
    # return total_absolute_error / total_samples if total_samples > 0 else -1
    return total_square_error / total_samples if total_samples > 0 else -1

In [ ]:
TIME_STEPS = 540
MIN_TIME_BIN_SIZE = 30
def objective(trial, params):
    n_bins = trial.suggest_int("n_time_bins", 2, 10)
    params['N_TIME_BINS'] = n_bins
    
    sizes = []
    current_sum = 0    
    
    for i in range(n_bins - 1):
        max_for_this_bin = TIME_STEPS - current_sum - (n_bins - 1 - i) * MIN_TIME_BIN_SIZE
        s = trial.suggest_int(f"size_bin_{i}", MIN_TIME_BIN_SIZE, max_for_this_bin, step=5)
        sizes.append(s)
        current_sum += s
    
    sizes.append(TIME_STEPS - current_sum)
    params['SIZES'] = sizes

    # The rest of the tuning params:
    params['N_EST_BT'] = trial.suggest_int("n_est_bt", 5, 60)
    params['MAX_DEPTH_BT'] = trial.suggest_int("max_depth_bt", 3, 8)
    params['N_EST_RT'] = trial.suggest_int("n_est_rt", 50, 5_000, step=50)
    params['MAX_DEPTH_RT'] = trial.suggest_int("max_depth_rt", 3, 10)
    params['MIN_CHILD_WEIGHT'] = trial.suggest_int("min_child_weight", 20, 200, step=10)
    params['HUBER_SLOPE'] = trial.suggest_float("huber_slope", 1, 10)
    params['BASE_SCORE_MULTIPLIER'] = trial.suggest_float("base_score_multiplier", 0, 3)
    params['EARLY_STOPPING'] = trial.suggest_int("early_stopping", 10, 200, step=10)

    score = train(params)
    
    return score
    

In [ ]:
my_objective = lambda trial: objective(trial,params)

study = opt.create_study(
    direction="minimize",
    sampler=opt.samplers.TPESampler(multivariate=True, warn_independent_sampling=False)
)

study.optimize(my_objective, n_trials=50)

print("Best Trial Score:", study.best_trial.value)
print("Best Params:", study.best_params)

### -------------------------------------------------------------

### export params

In [ ]:
export_best_params(study,using='MSE',filename="/home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/Models/Second_Model/optuna_history_Pred_yWide.txt")

### export params

### -------------------------------------------------------------

In [ ]:
def evauate_model(params):
    TREES = []
    TIME_STEPS = 540
    
    SIZES = params['sizes']
    if len(SIZES) != params['n_time_bins']:
        raise ValueError('The length of the SIZES must the number of time bins.')
    ENDS = np.cumsum(SIZES)
    
    clf_TRAINING_X = []
    clf_VALIDATION_X = []
    clf_TRAINING_Y = []
    clf_VALIDATION_Y = []
    reg_TRAINING_X = []
    reg_VALIDATION_X = []
    reg_TRAINING_Y = []
    reg_VALIDATION_Y = []
    DY_wide = y_wide - y_wide.groupby(level='date_id').shift(6)
    
    FITS_CLF = [] # To store the classifiers
    FITS_REG = [] # To store the regressors
    
    start_tracker = 0
    total_absolute_error = 0
    total_samples = 0

    #COMMENT TO REMEMBER, I THINK AS WRITTEN THE TIME SLICING MISSES THE LAST TIME POINT, WE SHOULDNT DO THAT
    #did i fix that already? lol
    for E in ENDS:
        T_start = start_tracker
        T_end = E
        start_tracker = E
        
        X_clf_train = x_new_feats.loc[pdidx[:end_clf_train,T_start:T_end-1],:].reset_index().drop(columns=['date_id'])
        Y_clf_train = DY_wide.loc[pdidx[:end_clf_train,T_start:T_end-1],:].values
        # Y_clf_train = y_wide.loc[pdidx[:end_clf_train,T_start:T_end-1],:].values
    
        X_clf_valid = x_new_feats.loc[pdidx[end_clf_train+1:,T_start:T_end-1],:].reset_index().drop(columns=['date_id'])
        Y_clf_valid = DY_wide.loc[pdidx[end_clf_train+1:,T_start:T_end-1],:].values
        # Y_clf_valid = y_wide.loc[pdidx[end_clf_train+1:,T_start:T_end-1],:].values
    
        train_mask = ~np.isnan(Y_clf_train)
        valid_mask = ~np.isnan(Y_clf_valid)
        
        ytr_bin = (Y_clf_train[train_mask] >= 0).astype(int)
        yva_bin = (Y_clf_valid[valid_mask] >= 0).astype(int)
    
        clf_TRAINING_X.append(X_clf_train[train_mask])
        clf_VALIDATION_X.append(X_clf_valid[valid_mask])
        
        clf_TRAINING_Y.append(ytr_bin)
        clf_VALIDATION_Y.append(yva_bin)
    
        
        BT = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            learning_rate=0.05,
            n_estimators=params['n_est_bt'],
            max_depth=params['max_depth_bt'],
        )
        BT.set_params(eval_metric=["auc", "logloss"])
        BT.fit(clf_TRAINING_X[-1], clf_TRAINING_Y[-1], eval_set=[(clf_VALIDATION_X[-1], clf_VALIDATION_Y[-1])], verbose=False)
        FITS_CLF.append(BT)
    
        X_reg_train = x_new_feats.loc[pdidx[end_clf_train+1:end_reg_train,T_start:T_end-1,:]].reset_index().drop(columns=['date_id'])
        # Y_reg_train = DY_wide.loc[pdidx[end_clf_train+1:end_reg_train,T_start:T_end-1,:]].values
        Y_reg_train = y_wide.loc[pdidx[end_clf_train+1:end_reg_train,T_start:T_end-1,:]].values
    
        X_reg_valid = x_new_feats.loc[pdidx[end_reg_train+1:,T_start:T_end-1,:]].reset_index().drop(columns=['date_id'])
        # Y_reg_valid = DY_wide.loc[pdidx[end_reg_train+1:,T_start:T_end-1,:]].values
        Y_reg_valid = y_wide.loc[pdidx[end_reg_train+1:,T_start:T_end-1,:]].values
    
        X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train)[:, 1]
        X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid)[:, 1]
    
        train_mask = ~np.isnan(Y_reg_train)
        valid_mask = ~np.isnan(Y_reg_valid)
        
        reg_TRAINING_X.append(X_reg_train.iloc[train_mask])
        reg_VALIDATION_X.append(X_reg_valid[valid_mask])
        
        reg_TRAINING_Y.append(Y_reg_train[train_mask])
        reg_VALIDATION_Y.append(Y_reg_valid[valid_mask])    
        
        
        RT = XGBRegressor(
            n_estimators=params['n_est_rt'],
            learning_rate=0.005,
            max_depth=params['max_depth_rt'],
            min_child_weight=params['min_child_weight'],
            objective='reg:pseudohubererror',
            huber_slope=params['huber_slope'],
            base_score=params['base_score_multiplier']*np.mean(reg_TRAINING_Y[-1]),
            tree_method='hist',
            early_stopping_rounds=params['early_stopping'],
        )
    
    
        fit = RT.fit(reg_TRAINING_X[-1], reg_TRAINING_Y[-1], eval_set=[(reg_VALIDATION_X[-1], reg_VALIDATION_Y[-1])], verbose=False)
        FITS_REG.append(fit)
        

        preds = RT.predict(reg_VALIDATION_X[-1])
        total_absolute_error += np.sum(np.abs(preds - reg_VALIDATION_Y[-1]))
        total_samples += len(preds)

    print('='*10 + ' TOTAL MAE ' + '='*10 )
    print(total_absolute_error/total_samples)
        
    return [FITS_CLF, clf_TRAINING_X, clf_VALIDATION_X, clf_TRAINING_Y, clf_VALIDATION_Y,
            FITS_REG, reg_TRAINING_X, reg_VALIDATION_X, reg_TRAINING_Y, reg_VALIDATION_Y]

In [ ]:
n_bins = study.best_params['n_time_bins']
sizes_list = []
best_params = study.best_params
last_size = 0
total_seconds_used = 0
for i in range(n_bins - 1):
    binstr = f'size_bin_{i}'
    bin_size = study.best_params[binstr]
    sizes_list.append(bin_size)
    total_seconds_used += bin_size
    best_params.pop(binstr)

sizes_list.append(540 - total_seconds_used)
                      
best_params['sizes'] = sizes_list
[FITS_CLF, clf_TRAINING_X, clf_VALIDATION_X, clf_TRAINING_Y, clf_VALIDATION_Y,
            FITS_REG, reg_TRAINING_X, reg_VALIDATION_X, reg_TRAINING_Y, reg_VALIDATION_Y] = evauate_model(best_params)

In [ ]:
def run_clf_diagnostics(bucket_idx, model, xtr, ytr_bin, xva, yva_bin):
    # 1. Generate Probabilities
    proba_train = model.predict_proba(xtr)[:, 1]
    proba_valid = model.predict_proba(xva)[:, 1]

    print(f"\n{'='*30} BUCKET {bucket_idx+1} CLASSIFIER {'='*30}")
    print(f"VAL AUC: {roc_auc_score(yva_bin, proba_valid):.5f} | LogLoss: {log_loss(yva_bin, proba_valid):.5f}")
    print(f"PR AUC:  {average_precision_score(yva_bin, proba_valid):.5f} | Brier:   {brier_score_loss(yva_bin, proba_valid):.5f}")

    # --- Plot 1: Probability Distribution (Histogram) ---
    fig, axH = plt.subplots(figsize=(10, 5))
    axH.hist(proba_valid[yva_bin == 0], bins=50, alpha=0.5, label='Actual Down', color='red', density=True)
    axH.hist(proba_valid[yva_bin == 1], bins=50, alpha=0.5, label='Actual Flat/Up', color='green', density=True)
    axH.axvline(0.5, color='black', linestyle='--', alpha=0.6)
    axH.set_title(f'Bucket {bucket_idx+1}: Probability Distribution (Validation)')
    axH.set_xlabel('Predicted Probabilities')
    axH.legend()

    # --- Plot 2: Scatter & Metrics Suite ---
    fig2, (axROC, axPR, axC) = plt.subplots(1, 3, figsize=(25, 5))

    # i added plot 3
    fig3, (axT, axV) = plt.subplots(1, 2, figsize=(8,5))
    
    # Scatters
    axT.scatter(range(len(proba_train)), proba_train, c=['g' if y==1 else 'k' for y in ytr_bin], s=2, alpha=0.3)
    axV.scatter(range(len(proba_valid)), proba_valid, c=['g' if y==1 else 'k' for y in yva_bin], s=2, alpha=0.3)
    axT.set_title("Train Probs"), axV.set_title("Valid Probs")
    
    # ROC
    fprV, tprV, _ = roc_curve(yva_bin, proba_valid)
    axROC.plot(fprV, tprV, color='green', label='Valid')
    axROC.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axROC.set_title("ROC Curve")

    # PR Curve
    precV, recV, _ = precision_recall_curve(yva_bin, proba_valid)
    axPR.step(recV, precV, color='green')
    axPR.set_title("Precision-Recall")

    # Calibration
    p_true, p_pred = calibration_curve(yva_bin, proba_valid, n_bins=10)
    axC.plot([0, 1], [0, 1], "k--")
    axC.plot(p_pred, p_true, "s-g")
    axC.set_title("Calibration")

    plt.tight_layout()
    plt.show()

# Execution
for i in range(len(FITS_CLF)):
    run_clf_diagnostics(i, FITS_CLF[i], clf_TRAINING_X[i], clf_TRAINING_Y[i], 
                        clf_VALIDATION_X[i], clf_VALIDATION_Y[i])

In [ ]:
def run_reg_diagnostics(bucket_idx, model, xva, yva):
    # 1. Predictions & Residuals
    preds = model.predict(xva)
    residuals = yva - preds
    mae = np.mean(np.abs(residuals))
    
    # 2. Setup Figure
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6))
    fig.suptitle(f"BUCKET {bucket_idx+1} REGRESSOR ANALYSIS", fontsize=16, fontweight='bold')

    # Plot A: Actual vs Predicted
    ax1.scatter(yva, preds, alpha=0.2, s=10, color='dodgerblue')
    lims = [yva.min(), yva.max()]
    ax1.plot(lims, lims, 'r--', lw=2)
    ax1.set_title(f"Pred vs Actual (MAE: {mae:.6f})")
    ax1.set_xlabel("Actual Target"), ax1.set_ylabel("Predicted Target")

    # Plot B: Error Distribution
    sns.histplot(residuals, bins=100, kde=True, ax=ax2, color='purple')
    ax2.axvline(0, color='black', linestyle='-')
    ax2.set_title("Residual Distribution (Error)")

    # Plot C: Feature Importance (The "Stacking" Proof)
    importances = pd.Series(model.feature_importances_, index=xva.columns)
    # Highlight clf_prob if it exists
    colors = ['orange' if idx == 'clf_prob' else 'skyblue' for idx in importances.sort_values().tail(15).index]
    importances.sort_values().tail(15).plot(kind='barh', ax=ax3, color=colors)
    ax3.set_title("Top 15 Features (Orange = Classifier Hint)")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    # T_start = xva['seconds_in_bucket'].min()
    # T_end = xva['seconds_in_bucket'].max()
    # print(f"DEBUG EVALUATION: Bin {T_start}-{T_end} | Samples: {len(residuals)} | SumAbsErr: {np.sum(np.abs(residuals))}")

# Execution
for i in range(len(FITS_REG)):
    run_reg_diagnostics(i, FITS_REG[i], reg_VALIDATION_X[i], reg_VALIDATION_Y[i])

    # print(f"DEBUG: FINAL TOTAL SAMPLES: {sum([len(reg_VALIDATION_Y[i]) for i in range(2)])}")

In [ ]:
def print_decile_table(y_true, y_prob):
    df = pd.DataFrame({'actual': y_true, 'prob': y_prob})
    df['bin'] = pd.qcut(df['prob'], 10, duplicates='drop')
    
    summary = df.groupby('bin', observed=True).agg(
        Count=('actual', 'count'),
        Mean_Prob=('prob', 'mean'),
        Win_Rate=('actual', 'mean')
    ).reset_index()
    
    print(summary.to_string(index=False))

for i in range(len(FITS_REG)):
    print(f"VALIDATION WIN-RATE BY CLASSIFIER CONFIDENCE (Bucket {i+1}):")
    print_decile_table(clf_VALIDATION_Y[i], FITS_CLF[i].predict_proba(clf_VALIDATION_X[i])[:, 1])

In [ ]:
def audit_feature_leakage(x_df, y_series):
    # 1. Align the data
    temp_df = x_df.copy()
    temp_df['TARGET_TO_PREDICT'] = y_series
    
    # 2. Calculate Pearson Correlation
    correlations = temp_df.corr()['TARGET_TO_PREDICT'].sort_values(ascending=False)
    
    # 3. Filter out the target itself
    correlations = correlations.drop(labels=['TARGET_TO_PREDICT'])
    
    print(f"{'='*30} LEAKAGE AUDIT {'='*30}")
    print(f"Top 10 Positively Correlated Features:")
    print(correlations.head(10))
    print(f"\nTop 10 Negatively Correlated Features:")
    print(correlations.tail(10))
    
    max_corr = correlations.abs().max()
    print(f" Max Correlation: {max_corr:.4f}")

# Run it on your first bucket
audit_feature_leakage(clf_TRAINING_X[0], clf_TRAINING_Y[0])

In [ ]:
true_errors = []
true_counts = []

for i in range(len(FITS_REG)):
    # 1. Get the Raw Data
    x_val = reg_VALIDATION_X[i]
    y_val = reg_VALIDATION_Y[i].flatten()
    
    # 2. Re-apply the mask (Crucial!)
    mask = ~np.isnan(y_val)
    x_val_clean = x_val.iloc[mask]
    y_val_clean = y_val[mask]
    
    # 3. Predict and sum ABSOLUTE error
    preds = FITS_REG[i].predict(x_val_clean)
    abs_err_sum = np.sum(np.abs(preds - y_val_clean))
    
    true_errors.append(abs_err_sum)
    true_counts.append(len(y_val_clean))
    
    print(f"Bin {i+1}: Real MAE = {abs_err_sum/len(y_val_clean):.6f} (Count: {len(y_val_clean)})")

print(f"\nFINAL VERIFIED MAE: {sum(true_errors) / sum(true_counts)}")

# ---------------------------------------------------------------------------------------------------------------

# ---------------------------------------------------------------------------------------------------------------

# ---------------------------------------------------------------------------------------------------------------

# LOOPING THIS WORKFLOR OVER ALL STOCKS

In [2]:
def preprocess_ALL_STOCKS(df):
    df = (df
          .set_index(['date_id', 'stock_id', 'seconds_in_bucket'])
          .sort_index(level=['date_id', 'stock_id', 'seconds_in_bucket']))
    
    df = df.drop(['row_id', 'time_id'], axis=1)

    targets_stock = df[['target']]
    df = df.drop(['target'], axis=1)

    return df, targets_stock

print(f"LOADING ALL STOCKS INTO x_wide, y_wide. YOU HAVE 5 SECONDS TO ABORT")
for i in range(5):
    print(f"{i+1}...")
    T.sleep(1)

path_to_data = "/home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/train.csv"
print(f"... Loading data from {path_to_data} into ALL_DATA...")
ALL_DATA = pd.read_csv(path_to_data)
x_wide, y_wide = preprocess_ALL_STOCKS(ALL_DATA)
%reset_selective ALL_ASSETS

LOADING ALL STOCKS INTO x_wide, y_wide. YOU HAVE 5 SECONDS TO ABORT
1...
2...
3...
4...
5...
... Loading data from /home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/train.csv into ALL_DATA...


Once deleted, variables cannot be recovered. Proceed (y/[n])?   y


In [3]:
def log_and_clean(df: pd.DataFrame, cols):
    df = df.copy()
    with np.errstate(divide='ignore', invalid='ignore'):
        df[cols] = np.log(df[cols])
    df[cols] = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df

RAW_SIZE_COLS = ['bid_size', 'ask_size', 'matched_size', 'imbalance_size', 'bid_ask_lot_spread']
RAW_PRICE_COLS = ['wap']
BASE_LOG_TARGETS = RAW_SIZE_COLS + RAW_PRICE_COLS
LAG_LOG_TARGETS = [f"{c}_lag_6" for c in BASE_LOG_TARGETS]

LOG_COLS = BASE_LOG_TARGETS + LAG_LOG_TARGETS + ['wap_frac'] # wap_frac should not get lagged, it already contains a lag

In [4]:
def Feature_Construction(x_wide_stock):

    # --- 1. Feature Construction ---
    bid_p, ask_p = x_wide_stock['bid_price'], x_wide_stock['ask_price']
    bid_s, ask_s = x_wide_stock['bid_size'], x_wide_stock['ask_size']
    
    rel_wap = (bid_p * ask_s - ask_p * bid_s) / (bid_s + ask_s)
    rel_wap.name = 'relative_wap'
    
    log_rel_wap = rel_wap.copy()
    log_rel_wap.name = 'log_relative_wap'
    
    log_wap = x_wide_stock['wap'].copy()
    log_wap.name = 'log_wap'
    
    BAspread = x_wide_stock['bid_size'] - x_wide_stock['ask_size']
    BAspread.name = 'bid_ask_lot_spread'
    
    x_almost = pd.concat([x_wide_stock, rel_wap, log_rel_wap, log_wap, BAspread], axis=1)
    
    # --- 3. Identify features for rolling/lagging ---
    exclude = ['seconds_in_bucket']
    features_to_process = [c for c in x_almost.columns if c not in exclude]
    
    # --- 4. Rolling medians (window=6) ---
    roll_df = (
        x_almost.groupby('date_id')[features_to_process]
                .rolling(window=6)
                .median()
                .reset_index(level=0, drop=True)
    )
    roll_df.columns = [f"{c}_roll6" for c in roll_df.columns]
    
    # --- 5. Lag 6 only ---
    lagged = (
        x_almost.groupby('date_id')[features_to_process]
                .shift(6)
                .add_suffix('_lag_6')
    )
    
    Dwap = x_almost['wap'] - lagged['wap_lag_6']
    Dwap.name = 'delta_wap'
    
    wap_frac = x_almost['wap'] / lagged['wap_lag_6']
    wap_frac.name = 'wap_frac'
    
    log_wap_frac = wap_frac.copy()
    log_wap_frac.name = 'log_wap_frac'
    
    # --- 6. Final features ---
    x_new_feats = pd.concat([x_almost, Dwap, wap_frac, log_wap_frac, roll_df, lagged], axis=1)
    
    # --- Log scale large quantities ---
    x_new_feats = log_and_clean(x_new_feats, LOG_COLS)

    return x_new_feats


In [5]:
#######################
###  Default parameters
#######################
base_params = {
    'N_TIME_BINS' : 8,
    'SIZES' : [120,90,90,60,60,60,30,30],
    'N_EST_BT' : 20,
    'MAX_DEPTH_BT' : 6,
    'N_EST_RT' : 1_000,
    'MAX_DEPTH_RT' : 5,
    'MIN_CHILD_WEIGHT' : 100,
    'HUBER_SLOPE' : 5,
    'BASE_SCORE_MULTIPLIER' : 1,
    'EARLY_STOPPING' : 50
}
end_clf_train = 200
end_reg_train = 400

In [6]:
def train_per_stock(params,STOCK):
    TIME_STEPS = 540
    SIZES = params['SIZES']
    ENDS = np.cumsum(SIZES).tolist()

    x_STOCK = Feature_Construction(
        x_wide.loc[pdidx[:, STOCK, :], :].droplevel('stock_id')
    )
    
    y_STOCK = (
        y_wide.loc[pdidx[:, STOCK, :], :]
              .droplevel('stock_id')
              .copy()
    )
    
    Dy_STOCK = y_STOCK - y_STOCK.groupby(level='date_id').shift(6)
    
    start_tracker = 0

    # total_absolute_error = 0
    total_square_error = 0
    total_samples = 0
    
    for E in ENDS:
        T_start = start_tracker
        T_end = E
        start_tracker = E + 1
        
        # --- CLASSIFIER DATA ---
        X_clf_train = x_STOCK.loc[pdidx[:end_clf_train, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        Y_clf_train = Dy_STOCK.loc[pdidx[:end_clf_train, T_start:T_end], :].values
    
        X_clf_valid = x_STOCK.loc[pdidx[end_clf_train+1:, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        Y_clf_valid = Dy_STOCK.loc[pdidx[end_clf_train+1:, T_start:T_end], :].values
    
        train_mask = ~np.isnan(Y_clf_train).flatten()
        valid_mask = ~np.isnan(Y_clf_valid).flatten()
        
        ytr_bin = (Y_clf_train.flatten()[train_mask] >= 0).astype(int)
        yva_bin = (Y_clf_valid.flatten()[valid_mask] >= 0).astype(int)

        # --- CLASSIFIER SAFETY ---
        if len(ytr_bin) == 0 or len(yva_bin) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Classifier data.")
            raise opt.exceptions.TrialPruned()
    
        # --- FIT CLASSIFIER ---
        BT = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            learning_rate=0.05,
            n_estimators=params['N_EST_BT'],
            max_depth=params['MAX_DEPTH_BT'],
            eval_metric=["auc", "logloss"]
        )
        BT.fit(X_clf_train.iloc[train_mask], ytr_bin, 
               eval_set=[(X_clf_valid.iloc[valid_mask], yva_bin)], 
               verbose=False)
    
        # --- REGRESSOR DATA ---
        X_reg_train = x_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        # Y_reg_train = Dy_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].values
        Y_reg_train = y_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].values
    
        X_reg_valid = x_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        # Y_reg_valid = Dy_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].values
        Y_reg_valid = y_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].values
    
        # Apply Classifier probabilities as a feature
        X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train)[:, 1]
        X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid)[:, 1]
    
        reg_train_mask = ~np.isnan(Y_reg_train).flatten()
        reg_valid_mask = ~np.isnan(Y_reg_valid).flatten()
        
        # --- FIT REGRESSOR ---
        y_reg_tr_clean = Y_reg_train.flatten()[reg_train_mask]
        y_reg_vld_clean = Y_reg_valid.flatten()[reg_valid_mask]

        if len(y_reg_tr_clean) == 0 or len(y_reg_vld_clean) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Regressor data.")
            raise opt.exceptions.TrialPruned()

        current_base_score = np.mean(y_reg_tr_clean) * params['BASE_SCORE_MULTIPLIER']
        RT = xgb.XGBRegressor(
            n_estimators=params['N_EST_RT'],
            learning_rate=0.005,
            max_depth=params['MAX_DEPTH_RT'],
            min_child_weight=params['MIN_CHILD_WEIGHT'],
            objective='reg:pseudohubererror',
            huber_slope=params['HUBER_SLOPE'],
            base_score=current_base_score,
            tree_method='hist',
            early_stopping_rounds=params['EARLY_STOPPING'],
        )
    
        RT.fit(X_reg_train.iloc[reg_train_mask], y_reg_tr_clean, 
               eval_set=[(X_reg_valid.iloc[reg_valid_mask], Y_reg_valid.flatten()[reg_valid_mask])], 
               verbose=False)
        
        # --- ACCUMULATE ERROR ---
        preds = RT.predict(X_reg_valid.iloc[reg_valid_mask])
        actuals = Y_reg_valid.flatten()[reg_valid_mask]
        
        # total_absolute_error += np.sum(np.abs(preds - actuals))
        total_square_error += np.sum(np.square(preds - actuals))
        total_samples += len(actuals)

    # SQUARED ERROR!!!!!!
    # return total_absolute_error / total_samples if total_samples > 0 else -1
    return total_square_error / total_samples if total_samples > 0 else -1

In [7]:
TIME_STEPS = 540
MIN_TIME_BIN_SIZE = 30
def objective_per_stock(trial, params, STOCK):
    params = base_params.copy()
    n_bins = trial.suggest_int("n_time_bins", 2, 10)
    # n_bins = 2
    params['N_TIME_BINS'] = n_bins
    
    sizes = []
    current_sum = 0    
    
    for i in range(n_bins - 1):
        max_for_this_bin = TIME_STEPS - current_sum - (n_bins - 1 - i) * MIN_TIME_BIN_SIZE
        s = trial.suggest_int(f"size_bin_{i}", MIN_TIME_BIN_SIZE, max_for_this_bin, step=5)
        sizes.append(s)
        current_sum += s
    
    sizes.append(TIME_STEPS - current_sum)
    # sizes = [trial.suggest_int("size_bin_1", 400, 510, step=5)]
    # sizes.append(TIME_STEPS - sizes[0])    
    params['SIZES'] = sizes
    # print(sizes)
    

    # The rest of the tuning params:
    params['N_EST_BT'] = trial.suggest_int("n_est_bt", 5, 60)
    params['MAX_DEPTH_BT'] = trial.suggest_int("max_depth_bt", 3, 8)
    params['N_EST_RT'] = trial.suggest_int("n_est_rt", 50, 5_000, step=50)
    params['MAX_DEPTH_RT'] = trial.suggest_int("max_depth_rt", 3, 10)
    params['MIN_CHILD_WEIGHT'] = trial.suggest_int("min_child_weight", 20, 200, step=10)
    params['HUBER_SLOPE'] = trial.suggest_float("huber_slope", 1, 10)
    params['BASE_SCORE_MULTIPLIER'] = trial.suggest_float("base_score_multiplier", 0, 3)
    params['EARLY_STOPPING'] = trial.suggest_int("early_stopping", 10, 200, step=10)

    score = train_per_stock(params,STOCK)
    
    return score
    

In [12]:
def export_all_stocks(study_dict, filename="All_Stocks_Tuning"):
    # FILE NAME INPUT ONLY CARES ABOUT THE NAME OF THE TEXT FILE ITSELF WITHOUT THE .txt
    # AUTOMATICALLY SAVED AS TEXT FILE AND IN THE APPROPRIATE FOLDER
    FULL_PATH = "/home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/Models/Second_Model/"
    if '.' in filename:
        raise ValueError(f"Invalid file name. No extension necessary. File will be save to {FULL_PATH} as text file.")
        
    timestamp = datetime.now().strftime("%Y-%m-%d__%H:%M:%S")
    final_path = FULL_PATH + filename + "_" + timestamp + ".txt"

    print(f"Writing to\n{final_path}\n ........")
    for STOCK, STUDY in study_dict.items():
        completed_trials = Study_For_Stocks[STOCK].get_trials(states=(opt.trial.TrialState.COMPLETE,))
        if len(completed_trials) == 0:
            print(f"Skipping Stock {STOCK}, which has no completed trials..")
            continue
        best_params = STUDY.best_params
        best_value = STUDY.best_value

    
        # THE MODEL ACTUALLY TAKES IN A LIST OF SIZES, SO HERE WE CREATE IT AND DELETE THE REDUNDANT PARAMS
        n_bins = STUDY.best_params['n_time_bins']
        sizes_list = []
        best_params = STUDY.best_params
        last_size = 0
        total_seconds_used = 0
        for i in range(n_bins - 1):
            binstr = f'size_bin_{i}' #NEED TO CHANGE LATER, RIGHT NOW IS 4 BECAUSE FUCK ME
            bin_size = STUDY.best_params[binstr]
            sizes_list.append(bin_size)
            total_seconds_used += bin_size
            best_params.pop(binstr)
        
        sizes_list.append(540 - total_seconds_used)
                              
        best_params['sizes'] = sizes_list
        
        entry = {
            "STOCK": STOCK,
            "best_mae": best_value,
            "params": best_params
        }
        
        with open(final_path, "a") as f:
            f.write(f"STOCK NUMBER: {STOCK} \n")
            f.write(json.dumps(entry) + "\n")
            f.flush()
            os.fsync(f.fileno())
    print("Done!")

# RUNNING LOOP BELOW

In [9]:
import traceback

Study_For_Stocks = {}

for STOCK in range(1,199):
    print(f"RUNNING STOCK NUMBER {STOCK} ...")
    try:
        my_objective = lambda trial: objective_per_stock(trial,base_params,STOCK)
        study = opt.create_study(
            direction="minimize",
            sampler=opt.samplers.TPESampler(multivariate=True, warn_independent_sampling=False)
        )
        study.optimize(my_objective, n_trials=30)
        Study_For_Stocks[STOCK] = study
        
        print(f"Best Trial Score for STOCK {STOCK}: ", study.best_trial.value)
        print(f"Best Params STOCK {STOCK}: ", study.best_params)
    except Exception as E:
        print(f"STOCK {STOCK} failed.")
        print(E)
        print(traceback.format_exc())
        print('MOVING ON...')

/home/travis/.local/lib/python3.8/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-03-18 00:09:21,397] A new study created in memory with name: no-name-f42f2cd8-51df-49df-83fd-9ba20973f773


RUNNING STOCK NUMBER 1 ...


[I 2026-03-18 00:09:25,161] Trial 0 finished with value: 108.28894375544998 and parameters: {'n_time_bins': 2, 'size_bin_0': 310, 'n_est_bt': 47, 'max_depth_bt': 4, 'n_est_rt': 3650, 'max_depth_rt': 9, 'min_child_weight': 140, 'huber_slope': 1.628537645172498, 'base_score_multiplier': 1.9981224725536548, 'early_stopping': 130}. Best is trial 0 with value: 108.28894375544998.
[I 2026-03-18 00:09:29,856] Trial 1 finished with value: 111.05111827400168 and parameters: {'n_time_bins': 3, 'size_bin_0': 235, 'size_bin_1': 60, 'n_est_bt': 10, 'max_depth_bt': 8, 'n_est_rt': 4050, 'max_depth_rt': 8, 'min_child_weight': 50, 'huber_slope': 5.443554866512371, 'base_score_multiplier': 2.937496124813252, 'early_stopping': 50}. Best is trial 0 with value: 108.28894375544998.
[I 2026-03-18 00:09:32,443] Trial 2 finished with value: 108.67591027712807 and parameters: {'n_time_bins': 2, 'size_bin_0': 205, 'n_est_bt': 50, 'max_depth_bt': 4, 'n_est_rt': 4900, 'max_depth_rt': 4, 'min_child_weight': 20, 'hu

Skipping bin 0-30: No Classifier data.


[I 2026-03-18 00:11:15,392] Trial 21 finished with value: 108.99707053653547 and parameters: {'n_time_bins': 2, 'size_bin_0': 195, 'n_est_bt': 53, 'max_depth_bt': 3, 'n_est_rt': 4550, 'max_depth_rt': 4, 'min_child_weight': 30, 'huber_slope': 2.079991176504449, 'base_score_multiplier': 0.25394747060114936, 'early_stopping': 10}. Best is trial 0 with value: 108.28894375544998.
[I 2026-03-18 00:11:19,863] Trial 22 finished with value: 110.52015873512188 and parameters: {'n_time_bins': 4, 'size_bin_0': 225, 'size_bin_1': 210, 'size_bin_2': 70, 'n_est_bt': 48, 'max_depth_bt': 4, 'n_est_rt': 4400, 'max_depth_rt': 6, 'min_child_weight': 20, 'huber_slope': 6.9582597324150735, 'base_score_multiplier': 0.8926280842115343, 'early_stopping': 40}. Best is trial 0 with value: 108.28894375544998.
[I 2026-03-18 00:11:25,362] Trial 23 finished with value: 110.00262002679244 and parameters: {'n_time_bins': 3, 'size_bin_0': 155, 'size_bin_1': 275, 'n_est_bt': 60, 'max_depth_bt': 6, 'n_est_rt': 3800, 'max

...

Skipping bin 0-40: No Classifier data.


[I 2026-03-18 09:30:03,606] Trial 4 finished with value: 27.071314048625702 and parameters: {'n_time_bins': 10, 'size_bin_0': 135, 'size_bin_1': 65, 'size_bin_2': 55, 'size_bin_3': 30, 'size_bin_4': 100, 'size_bin_5': 35, 'size_bin_6': 30, 'size_bin_7': 30, 'size_bin_8': 30, 'n_est_bt': 60, 'max_depth_bt': 7, 'n_est_rt': 3000, 'max_depth_rt': 7, 'min_child_weight': 120, 'huber_slope': 6.4478699006912406, 'base_score_multiplier': 1.9579900380336657, 'early_stopping': 110}. Best is trial 0 with value: 26.14306701316619.
[I 2026-03-18 09:30:08,153] Trial 5 finished with value: 26.853849871431546 and parameters: {'n_time_bins': 2, 'size_bin_0': 410, 'n_est_bt': 52, 'max_depth_bt': 7, 'n_est_rt': 800, 'max_depth_rt': 6, 'min_child_weight': 130, 'huber_slope': 6.247045736179904, 'base_score_multiplier': 1.8959720957551358, 'early_stopping': 50}. Best is trial 0 with value: 26.14306701316619.
[I 2026-03-18 09:30:08,618] Trial 6 pruned. 


Skipping bin 0-40: No Classifier data.


[I 2026-03-18 09:30:12,731] Trial 7 finished with value: 26.30990882896734 and parameters: {'n_time_bins': 2, 'size_bin_0': 100, 'n_est_bt': 17, 'max_depth_bt': 7, 'n_est_rt': 2750, 'max_depth_rt': 4, 'min_child_weight': 40, 'huber_slope': 4.490010546146901, 'base_score_multiplier': 1.435722391082693, 'early_stopping': 180}. Best is trial 0 with value: 26.14306701316619.
[I 2026-03-18 09:30:17,195] Trial 8 finished with value: 26.186574957570123 and parameters: {'n_time_bins': 9, 'size_bin_0': 250, 'size_bin_1': 45, 'size_bin_2': 55, 'size_bin_3': 30, 'size_bin_4': 40, 'size_bin_5': 30, 'size_bin_6': 30, 'size_bin_7': 30, 'n_est_bt': 49, 'max_depth_bt': 6, 'n_est_rt': 4200, 'max_depth_rt': 4, 'min_child_weight': 80, 'huber_slope': 2.8264151198392353, 'base_score_multiplier': 0.5112495892164345, 'early_stopping': 30}. Best is trial 0 with value: 26.14306701316619.
[I 2026-03-18 09:30:24,521] Trial 9 finished with value: 26.924921692838375 and parameters: {'n_time_bins': 7, 'size_bin_0':

Best Trial Score for STOCK 198:  26.11571813253668
Best Params STOCK 198:  {'n_time_bins': 8, 'size_bin_0': 265, 'size_bin_1': 40, 'size_bin_2': 60, 'size_bin_3': 45, 'size_bin_4': 40, 'size_bin_5': 30, 'size_bin_6': 30, 'n_est_bt': 28, 'max_depth_bt': 3, 'n_est_rt': 2650, 'max_depth_rt': 5, 'min_child_weight': 70, 'huber_slope': 1.5805181809459379, 'base_score_multiplier': 0.06916477232501557, 'early_stopping': 30}


In [10]:
total = 0
for stock,study in Study_For_Stocks.items():
    if stock == 102:
        continue
    total+=study.best_value

print(total/(len(Study_For_Stocks)))
    

78.15974260703993


### -------------------------------------------------------------

### export params

In [13]:
export_all_stocks(Study_For_Stocks, filename='All_Stocks_Tuning_Y_TARGET_FIT')

Writing to
/home/travis/Documents/Reseach/Quant_Prep/kaggle/TradingAtTheClose/Workspace/Models/Second_Model/All_Stocks_Tuning_Y_TARGET_FIT_2026-03-18__10:14:08.txt
 ........
Skipping Stock 102, which has no completed trials..
Done!


### export params

### -------------------------------------------------------------

## Evaluate with each stock's best params and calculate MAE

In [15]:
def eval_per_stock(params,STOCK):
    TIME_STEPS = 540

    sizes = []
    current_sum = 0    

    n_bins = params['n_time_bins']
    for i in range(n_bins - 1):
        max_for_this_bin = TIME_STEPS - current_sum - (n_bins - 1 - i) * MIN_TIME_BIN_SIZE
        s = params[f"size_bin_{i}"]
        sizes.append(s)
        current_sum += s
    
    sizes.append(TIME_STEPS - current_sum)
    ENDS = np.cumsum(sizes)

    x_STOCK = Feature_Construction(
        x_wide.loc[pdidx[:, STOCK, :], :].droplevel('stock_id')
    )
    
    y_STOCK = (
        y_wide.loc[pdidx[:, STOCK, :], :]
              .droplevel('stock_id')
              .copy()
    )
    
    Dy_STOCK = y_STOCK - y_STOCK.groupby(level='date_id').shift(6)
    
    start_tracker = 0

    all_resids_train = []
    all_resids_valid = []
    
    for E in ENDS:
        T_start = start_tracker
        T_end = E
        start_tracker = E + 1
        
        # --- CLASSIFIER DATA ---
        X_clf_train = x_STOCK.loc[pdidx[:end_clf_train, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        Y_clf_train = Dy_STOCK.loc[pdidx[:end_clf_train, T_start:T_end], :].values
    
        X_clf_valid = x_STOCK.loc[pdidx[end_clf_train+1:, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        Y_clf_valid = Dy_STOCK.loc[pdidx[end_clf_train+1:, T_start:T_end], :].values
    
        train_mask = ~np.isnan(Y_clf_train).flatten()
        valid_mask = ~np.isnan(Y_clf_valid).flatten()
        
        ytr_bin = (Y_clf_train.flatten()[train_mask] >= 0).astype(int)
        yva_bin = (Y_clf_valid.flatten()[valid_mask] >= 0).astype(int)

        # --- CLASSIFIER SAFETY ---
        if len(ytr_bin) == 0 or len(yva_bin) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Classifier data.")
            raise opt.exceptions.TrialPruned()
    
        # --- FIT CLASSIFIER ---
        BT = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            learning_rate=0.05,
            n_estimators=params['n_est_bt'],
            max_depth=params['max_depth_bt'],
            eval_metric=["auc", "logloss"]
        )
        BT.fit(X_clf_train.iloc[train_mask], ytr_bin, 
               eval_set=[(X_clf_valid.iloc[valid_mask], yva_bin)], 
               verbose=False)
    
        # --- REGRESSOR DATA ---
        X_reg_train = x_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        # Y_reg_train = Dy_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].values
        Y_reg_train = y_STOCK.loc[pdidx[end_clf_train+1:end_reg_train, T_start:T_end], :].values
    
        X_reg_valid = x_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].reset_index().drop(columns=['date_id'])
        # Y_reg_valid = Dy_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].values
        Y_reg_valid = y_STOCK.loc[pdidx[end_reg_train+1:, T_start:T_end], :].values
    
        # Apply Classifier probabilities as a feature
        X_reg_train["clf_prob"] = BT.predict_proba(X_reg_train)[:, 1]
        X_reg_valid["clf_prob"] = BT.predict_proba(X_reg_valid)[:, 1]
    
        reg_train_mask = ~np.isnan(Y_reg_train).flatten()
        reg_valid_mask = ~np.isnan(Y_reg_valid).flatten()
        
        # --- FIT REGRESSOR ---
        y_reg_tr_clean = Y_reg_train.flatten()[reg_train_mask]
        y_reg_vld_clean = Y_reg_valid.flatten()[reg_valid_mask]

        if len(y_reg_tr_clean) == 0 or len(y_reg_vld_clean) == 0:
            print(f"Skipping bin {T_start}-{T_end}: No Regressor data.")
            raise opt.exceptions.TrialPruned()

        current_base_score = np.mean(y_reg_tr_clean) * params['base_score_multiplier']
        RT = xgb.XGBRegressor(
            n_estimators=params['n_est_rt'],
            learning_rate=0.005,
            max_depth=params['max_depth_rt'],
            min_child_weight=params['min_child_weight'],
            objective='reg:pseudohubererror',
            huber_slope=params['huber_slope'],
            base_score=current_base_score,
            tree_method='hist',
            early_stopping_rounds=params['early_stopping'],
        )
    
        RT.fit(X_reg_train.iloc[reg_train_mask], y_reg_tr_clean, 
               eval_set=[(X_reg_valid.iloc[reg_valid_mask], Y_reg_valid.flatten()[reg_valid_mask])], 
               verbose=False)
        
        # --- ACCUMULATE ERROR ---
        preds = RT.predict(X_reg_valid.iloc[reg_valid_mask])
        actuals = Y_reg_valid.flatten()[reg_valid_mask]
        all_resids_valid.extend(np.abs(preds - actuals))

        # --- ERROR OVER TRAINING DATA
        preds = RT.predict(X_reg_train.iloc[reg_train_mask])
        actuals = Y_reg_train.flatten()[reg_train_mask]
        all_resids_train.extend(np.abs(preds - actuals))

    return all_resids_valid, all_resids_train

In [16]:
all_resids_train = []
all_resids_valid = []

for stock,study in Study_For_Stocks.items():
    if stock == 102:
        continue
    print(f'Calculating stock {stock} CONTRIBUTION...')
    validExt, trainExt = eval_per_stock(study.best_params, stock)
    all_resids_train.extend(trainExt)
    all_resids_valid.extend(validExt)
    print('SO FAR --- MAE over each individual point - TRAINING:',np.mean(all_resids_train))
    print('SO FAR --- MAE over each individual point - VALIDATION:',np.mean(all_resids_valid))

print('FINAL --- MAE over each individual point - TRAINING:',np.mean(all_resids_train))
print('FINAL --- MAE over each individual point - VALIDATION:',np.mean(all_resids_valid))

Calculating stock 1 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 8.845377982145342
SO FAR --- MAE over each individual point - VALIDATION: 7.799244305513649
Calculating stock 2 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 8.032165890534424
SO FAR --- MAE over each individual point - VALIDATION: 7.6175987286835
Calculating stock 3 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 6.602004972264778
SO FAR --- MAE over each individual point - VALIDATION: 6.216493729967969
Calculating stock 4 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 6.007222985726369
SO FAR --- MAE over each individual point - VALIDATION: 5.62425923037559
Calculating stock 5 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 6.291891637191684
SO FAR --- MAE over each individual point - VALIDATION: 5.927724267716468
Calculating stock 6 CONTRIBUTION...
SO FAR --- MAE over each individual point - TRAINING: 6.2

# First go:   
FINAL --- MAE over each individual point - TRAINING: 7.234585310743236   
FINAL --- MAE over each individual point - VALIDATION: 7.112885661330015   

# Second go (forcing 2 time bins):   
FINAL --- MAE over each individual point - TRAINING: 7.281736538983243   
FINAL --- MAE over each individual point - VALIDATION: 7.064507016004619

# Third go (predicting TARGET and NOT its CHANGE):   
FINAL --- MAE over each individual point - TRAINING: 6.156940350850401   
FINAL --- MAE over each individual point - VALIDATION: 5.928108244664085